In [ ]:
import pandas as pd
from langchain_community.llms import Ollama
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# 初始化 Ollama
llm = Ollama(
    model="llama3.1",
    temperature=0.75,
)

# 读取 CSV 文件
df = pd.read_csv("product_data.csv")

# 创建 agent 
agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True 
)

# 查询
#query = "Display the product_name of all items with a price greater than 100"
query = "tell me the price of Apple"
response = agent.run(query)

In [ ]:
import pandas as pd
from langchain_community.llms import Ollama
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_community.embeddings import OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# 1. 数据准备
df = pd.read_csv("product_data.csv")

# 2. 初始化组件
llm = Ollama(model="llama3.1", temperature=0.75)
embeddings = OllamaEmbeddings(
    model="bge-m3",
    base_url="http://localhost:11434"
)

# 3. 创建详细的文本描述
def create_detailed_text(df):
    texts = []
    for index, row in df.iterrows():
        text = f"Product Details - "
        for col in df.columns:
            text += f"{col}: {row[col]}. "
        texts.append(text)
    return texts

# 4. 准备文档
texts = create_detailed_text(df)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
splits = text_splitter.create_documents(texts)

# 5. 创建向量存储
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

# 6. 创建提示模板
template = """
基于以下信息回答问题：

上下文信息：
{context}

问题: {question}

如果数据中没有相关信息，请明确说明"数据库中没有这个产品的信息"。
请基于实际数据回答，不要猜测或编造信息。

回答：
"""

PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]  
)

# 7. 设置 QA 链
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": PROMPT
    }
)

# 8. 创建 agent
agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True
)

# 9. 创建查询函数
def enhanced_query(query, df, qa_chain, agent):
    try:
        # 使用 invoke 替代 __call__
        context = qa_chain.invoke({
            "query": query
        })
        
        # 结合 RAG 结果和 DataFrame 信息
        enhanced_prompt = f"""
        基于以下信息回答问题：
        1. 检索到的相关信息：{context['result']}
        2. 数据框架中的列：{list(df.columns)}
        
        问题：{query}
        """
        
        response = agent.run(enhanced_prompt)
        
        return {
            "answer": response,
            "context": context['source_documents']
        }
    except Exception as e:
        return {
            "error": str(e),
            "suggestion": "请检查查询内容是否与数据集相关"
        }

# 10. 查询函数
def query_product(product_name):
    query = f"请告诉我 {product_name} 的价格是多少？"
    result = enhanced_query(query, df, qa_chain, agent)
    
    if "error" in result:
        print(f"查询错误: {result['error']}")
        print(f"建议: {result['suggestion']}")
    else:
        print(f"回答: {result['answer']}")
        print("\n相关上下文:")
        for doc in result['context']:
            print(f"- {doc.page_content}")

# 使用示例
query_product("Apple")
